In [1]:
import pandas as pd
import numpy as np
import biogeme.database as db
import biogeme.biogeme as bio
from biogeme import models
from biogeme.expressions import Beta, bioDraws, log, exp, MonteCarlo, Variable, bioNormalCdf, Elem

# =============================================================================
# 1. DATA LOADING AND PRE-PROCESSING
# =============================================================================
# Load the synthetic dataset generated from the Monte Carlo simulation
df = pd.read_csv('synthetic_maas_dataset.csv')

# Perform linear standardization (Z-score normalization)
df['Age_norm'] = (df['Age'] - df['Age'].mean()) / df['Age'].std()
df['Income_norm'] = (df['Income'] - df['Income'].mean()) / df['Income'].std()

database = db.Database('MaaS_Synthetic', df)

# Define Biogeme Variables from the dataset columns
Choice = Variable('Choice')
Time_Car = Variable('Time_Car')
Cost_Car = Variable('Cost_Car')
Time_PT = Variable('Time_PT')
Cost_PT = Variable('Cost_PT')
Time_MaaS = Variable('Time_MaaS')
Cost_MaaS = Variable('Cost_MaaS')
Ind_1 = Variable('Ind_1')
Ind_2 = Variable('Ind_2')
Ind_3 = Variable('Ind_3')
Age_norm = Variable('Age_norm')
Income_norm = Variable('Income_norm')

# =============================================================================
# 2. STRUCTURAL EQUATION OF THE LATENT VARIABLE
# =============================================================================
# Defining the latent construct (Pro-MaaS attitude) with a linear specification

coef_intercept = Beta('coef_intercept', 0, None, None, 0)
coef_age = Beta('coef_age', 0, None, None, 0)
coef_income = Beta('coef_income', 0, None, None, 0)

# Latent Variable (LV) definition incorporating stochastic disturbance

LV = (
    coef_intercept 
    + coef_age * Age_norm 
    + coef_income * Income_norm 
    + bioDraws('omega', 'NORMAL_HALTON5')
)

# =============================================================================
# 3. MEASUREMENT EQUATIONS 
# =============================================================================
# --- 3.1 Intercepts and Factor Loadings ---
# Intercepts for measurement models 

INTER_I1 = Beta('INTER_I1', 0, None, None, 0) 
INTER_I2 = Beta('INTER_I2', 0, None, None, 0)
INTER_I3 = Beta('INTER_I3', 0, None, None, 0)

B_I1 = Beta('B_I1', 1.0, None, None, 0)       
B_I2 = Beta('B_I2', 1.0, None, None, 0)
B_I3 = Beta('B_I3', 1.0, None, None, 0)

MODEL_I1 = INTER_I1 + B_I1 * LV
MODEL_I2 = INTER_I2 + B_I2 * LV
MODEL_I3 = INTER_I3 + B_I3 * LV


# --- 3.2 Threshold Specification ---
# A 5-point Likert scale requires 4 cut-off points (thresholds)

delta_1 = Beta('delta_1', 0.5, 0.0001, None, 0) 
delta_2 = Beta('delta_2', 0.5, 0.0001, None, 0)

tau_1 = -delta_1 - delta_2
tau_2 = -delta_1
tau_3 = delta_1
tau_4 = delta_1 + delta_2

# --- 3.3 Construct Probabilities for Indicators (Example: Indicator 1) ---
I1_tau_1 = (tau_1 - MODEL_I1)
I1_tau_2 = (tau_2 - MODEL_I1)
I1_tau_3 = (tau_3 - MODEL_I1)
I1_tau_4 = (tau_4 - MODEL_I1)

Dict_Ind_1 = {
    1: bioNormalCdf(I1_tau_1),
    2: bioNormalCdf(I1_tau_2) - bioNormalCdf(I1_tau_1),
    3: bioNormalCdf(I1_tau_3) - bioNormalCdf(I1_tau_2),
    4: bioNormalCdf(I1_tau_4) - bioNormalCdf(I1_tau_3),
    5: 1 - bioNormalCdf(I1_tau_4),
}
P_I1 = Elem(Dict_Ind_1, Ind_1)


I2_tau_1 = (tau_1 - MODEL_I2)
I2_tau_2 = (tau_2 - MODEL_I2)
I2_tau_3 = (tau_3 - MODEL_I2)
I2_tau_4 = (tau_4 - MODEL_I2)

Dict_Ind_2 = {
    1: bioNormalCdf(I2_tau_1),
    2: bioNormalCdf(I2_tau_2) - bioNormalCdf(I2_tau_1),
    3: bioNormalCdf(I2_tau_3) - bioNormalCdf(I2_tau_2),
    4: bioNormalCdf(I2_tau_4) - bioNormalCdf(I2_tau_3),
    5: 1 - bioNormalCdf(I2_tau_4),
}
P_I2 = Elem(Dict_Ind_2, Ind_2)


I3_tau_1 = (tau_1 - MODEL_I3)
I3_tau_2 = (tau_2 - MODEL_I3)
I3_tau_3 = (tau_3 - MODEL_I3)
I3_tau_4 = (tau_4 - MODEL_I3)

Dict_Ind_3 = {
    1: bioNormalCdf(I3_tau_1),
    2: bioNormalCdf(I3_tau_2) - bioNormalCdf(I3_tau_1),
    3: bioNormalCdf(I3_tau_3) - bioNormalCdf(I3_tau_2),
    4: bioNormalCdf(I3_tau_4) - bioNormalCdf(I3_tau_3),
    5: 1 - bioNormalCdf(I3_tau_4),
}
P_I3 = Elem(Dict_Ind_3, Ind_3)


# =============================================================================
# 4. CHOICE MODEL
# =============================================================================
# Alternative Specific Constants
ASC_CAR  = Beta('ASC_CAR', 0, None, None, 1)  
ASC_PT   = Beta('ASC_PT', 0, None, None, 0)
ASC_MAAS = Beta('ASC_MAAS', 0, None, None, 0)

# Generic coefficients for travel time and cost
B_TIME   = Beta('B_TIME', 0, None, None, 0)
B_COST   = Beta('B_COST', 0, None, None, 0)

# Impact coefficient of the Latent Variable on the MaaS alternative utility
B_LV_MAAS = Beta('B_LV_MAAS', 1.0, None, None, 0)

# Utility Functions
V0 = ASC_CAR + B_TIME * Time_Car + B_COST * Cost_Car
V1 = ASC_PT + B_TIME * Time_PT + B_COST * Cost_PT
V2 = ASC_MAAS + B_TIME * Time_MaaS + B_COST * Cost_MaaS + B_LV_MAAS * LV 

V = {0: V0, 1: V1, 2: V2}
cond_prob_choice = models.logit(V, None, Choice)

# =============================================================================
# 5. JOINT LIKELIHOOD AND ESTIMATION
# =============================================================================

cond_like = cond_prob_choice * P_I1 * P_I2 * P_I3

like = MonteCarlo(cond_like)
logprob = log(like)


biogeme = bio.BIOGEME(database, logprob, number_of_draws=500)
biogeme.modelName = "ICLV_Synthetic_OrderedProbit"

print("Starting Biogeme estimation...")
results = biogeme.estimate()

print(results.shortSummary())
pandas_results = results.getEstimatedParameters()
print(pandas_results)

The number of draws (500) is low. The results may not be meaningful.


Starting Biogeme estimation...
Results for model ICLV_Synthetic_OrderedProbit
Nbr of parameters:		16
Sample size:			10000
Excluded data:			0
Final log likelihood:		-45433.54
Akaike Information Criterion:	90899.07
Bayesian Information Criterion:	91014.44

                   Value  Rob. Std err  Rob. t-test  Rob. p-value
ASC_MAAS       -1.808545      0.031677   -57.093804  0.000000e+00
ASC_PT         -0.455008      0.056667    -8.029430  8.881784e-16
B_COST         -0.195083      0.005414   -36.035305  0.000000e+00
B_I1            1.554762      0.020140    77.198340  0.000000e+00
B_I2            1.486598      0.020319    73.162541  0.000000e+00
B_I3            1.501786      0.020098    74.724956  0.000000e+00
B_LV_MAAS       2.867165      0.076643    37.409362  0.000000e+00
B_TIME         -0.077846      0.001752   -44.438534  0.000000e+00
INTER_I1       -0.016426      0.019749    -0.831757  4.055464e-01
INTER_I2       -0.015038      0.020049    -0.750053  4.532226e-01
INTER_I3       -0.0

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13712\187212760.py:156: DeprecationWarning: shortSummary is deprecated; use short_summary instead.
  print(results.shortSummary())
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13712\187212760.py:157: DeprecationWarning: getEstimatedParameters is deprecated; use get_estimated_parameters instead.
  pandas_results = results.getEstimatedParameters()
